<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-06-rag-systems/lesson-6.2-document-processing/notebooks/exercises-6_2.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 6.2: Document Processing Pipeline

*Module 6 · Retrieval-Augmented Generation (RAG)*

> RAG is only as good as its input. Feed it messy, poorly chunked documents and you get bad retrieval. Feed it clean, intelligently chunked documents and you get precise answers. This lesson builds the ingestion pipeline: load any format (PDF, HTML, Markdown), chunk with 3 strategies (fixed, recursive, semantic), and measure which works best.

`PDF/HTML/MD Loaders` · `3 Chunking Strategies` · `Comparison Metrics` · `60 min`

---

## About this notebook

This notebook contains the **4 runnable code examples** from the published lesson `lesson-6.2.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Document Loaders — Extract Text from Any Format — `01_loaders.py`
2. Step 2 — Three Chunking Strategies — Coded and Compared — `02_chunking_strategies.py`
3. Step 3 — Measuring Chunk Quality — Retrieval Accuracy — `03_chunk_eval.py`
4. Step 4 — Production Document Pipeline — Complete Class — `04_pipeline.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai numpy beautifulsoup4


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Document Loaders — Extract Text from Any Format

PDF, HTML, Markdown, plain text — one loader class to handle them all.

**`01_loaders.py`** · _Block 1 of 4_

DOCUMENT LOADERS — Extract text from any format — pip install pymupdf beautifulsoup4 markdown


In [ ]:
# DOCUMENT LOADERS — Extract text from any format
# pip install pymupdf beautifulsoup4 markdown
import re
from pathlib import Path

class DocumentLoader:
    """Load and extract text from PDF, HTML, Markdown, or plain text."""

    def load(self, path: str) -> dict:
        ext = Path(path).suffix.lower()
        loaders = {".pdf": self._pdf, ".html": self._html,
                   ".md": self._markdown, ".txt": self._text}
        loader = loaders.get(ext, self._text)
        text = loader(path)
        text = re.sub(r'\s+', ' ', text).strip()
        return {"source": path, "text": text, "words": len(text.split()), "format": ext}

    def _pdf(self, path):
        import fitz  # pymupdf
        doc = fitz.open(path)
        return " ".join(page.get_text() for page in doc)

    def _html(self, path):
        from bs4 import BeautifulSoup
        with open(path) as f: soup = BeautifulSoup(f.read(), "html.parser")
        [s.decompose() for s in soup(["script","style","nav","footer"])]
        return soup.get_text(" ")

    def _markdown(self, path):
        with open(path) as f: text = f.read()
        text = re.sub(r'#{1,6}\s+', '', text)  # remove headers markup
        text = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', text)  # links
        text = re.sub(r'[*_`~]+', '', text)  # formatting
        return text

    def _text(self, path):
        with open(path) as f: return f.read()

# Demo with simulated files
loader = DocumentLoader()
print("DocumentLoader supports: .pdf, .html, .md, .txt")
print("  PDF: pymupdf extracts text, tables, even images-as-text")
print("  HTML: BeautifulSoup strips scripts/nav/footer, keeps content")
print("  MD: regex removes #headers, [links](url), **bold**")
print("  All output: clean plaintext ready for chunking")


### Step 2 · Three Chunking Strategies — Coded and Compared

Fixed-size, recursive (split by structure), and semantic (split by meaning).

**`02_chunking_strategies.py`** · _Block 2 of 4_

THREE CHUNKING STRATEGIES — Coded and compared


In [ ]:
# THREE CHUNKING STRATEGIES — Coded and compared
import re
import numpy as np

# ── Strategy 1: Fixed-size chunks ──
def chunk_fixed(text, size=200, overlap=50):
    """Split every N words with overlap. Simple but dumb."""
    words = text.split()
    chunks = []
    for i in range(0, len(words), size - overlap):
        c = " ".join(words[i:i+size])
        if len(c.split()) >= 20: chunks.append(c)
    return chunks

# ── Strategy 2: Recursive (structure-aware) ──
def chunk_recursive(text, max_size=200, separators=None):
    """Split by paragraph, then sentence, then word. Respects structure."""
    separators = separators or ["\n\n", "\n", ". ", " "]
    chunks = []
    def _split(text, sep_idx=0):
        if len(text.split()) <= max_size:
            if len(text.split()) >= 15: chunks.append(text.strip())
            return
        if sep_idx >= len(separators):
            chunks.append(" ".join(text.split()[:max_size]))
            return
        parts = text.split(separators[sep_idx])
        current = ""
        for part in parts:
            if len((current + part).split()) <= max_size:
                current += separators[sep_idx] + part
            else:
                if current.strip(): _split(current.strip(), sep_idx + 1)
                current = part
        if current.strip(): _split(current.strip(), sep_idx + 1)
    _split(text)
    return chunks

# ── Strategy 3: Semantic (meaning-aware) ──
def chunk_semantic(text, threshold=0.75, embed_fn=None):
    """Group sentences by embedding similarity. New chunk on topic shift."""
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if len(s.split())>=3]
    if not embed_fn:  # fallback: word overlap similarity
        def embed_fn(t):
            words = set(t.lower().split())
            return words
    chunks, current = [], [sentences[0]]
    for i in range(1, len(sentences)):
        prev_words = set(" ".join(current[-2:]).lower().split())
        curr_words = set(sentences[i].lower().split())
        overlap = len(prev_words & curr_words) / max(len(prev_words | curr_words), 1)
        if overlap >= threshold * 0.3:  # same topic
            current.append(sentences[i])
        else:  # topic shift = new chunk
            chunks.append(" ".join(current))
            current = [sentences[i]]
    if current: chunks.append(" ".join(current))
    return chunks

# ── Compare all 3 on the same document ──
doc = """Netsetos is an educational technology company founded in 2024 in Hyderabad.
We offer courses in Generative AI, Data Science, and Cloud Computing.

Our refund policy allows full refunds within 7 days of purchase.
After 7 days, we offer a 50 percent refund up to 30 days.
After 30 days, no refunds are available.

The GenAI course costs 14999 rupees and includes lifetime access.
Students also get Discord community access and weekly live sessions.
Google Cloud credits worth 5000 rupees are included.

Live classes are held daily at 7 PM IST on YouTube.
Recordings are uploaded within 2 hours after the session.
The course completion rate is 85 percent with 4.8 star rating."""

print("Chunking Strategy Comparison:\n")
for name, fn in [("Fixed (200w)", lambda t: chunk_fixed(t,60,10)),
                  ("Recursive", lambda t: chunk_recursive(t,60)),
                  ("Semantic", lambda t: chunk_semantic(t))]:
    chunks = fn(doc)
    print(f"  {name}: {len(chunks)} chunks")
    for i,c in enumerate(chunks):
        print(f"    [{i}] ({len(c.split())}w) {c[:65]}...")
    print()


> **What just happened?** Three strategies on the same document: Fixed split every 60 words, cutting mid-paragraph. Recursive split by \n\n first, keeping paragraphs intact — refund policy stays in one chunk. Semantic grouped sentences by topic overlap — each chunk is about one subject. Recursive is the best default for most RAG systems. Semantic gives highest quality but is slower (needs embeddings).


### Step 3 · Measuring Chunk Quality — Retrieval Accuracy

Good chunks = good retrieval. We measure: does the right chunk get retrieved for each question?

**`03_chunk_eval.py`** · _Block 3 of 4_

CHUNKING EVALUATION — Which strategy retrieves best?


In [ ]:
# CHUNKING EVALUATION — Which strategy retrieves best?
import google.generativeai as genai, numpy as np, os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

def embed(text):
    return np.array(genai.embed_content(model="models/text-embedding-004",content=text)["embedding"])

def eval_retrieval(chunks, test_cases, k=1):
    """Test: does the correct chunk get retrieved for each question?"""
    embs = [embed(c) for c in chunks]
    correct = 0
    for question, expected_keyword in test_cases:
        qe = embed(question)
        scores = [(i, np.dot(qe,e)/(np.linalg.norm(qe)*np.linalg.norm(e))) for i,e in enumerate(embs)]
        scores.sort(key=lambda x:-x[1])
        top_chunk = chunks[scores[0][0]]
        if expected_keyword.lower() in top_chunk.lower():
            correct += 1
    return correct / len(test_cases)

# Test questions with expected keywords in the correct chunk
test_cases = [
    ("What is the refund policy?", "refund"),
    ("How much does the course cost?", "14999"),
    ("When are live classes?", "7 PM"),
    ("What is the completion rate?", "85"),
]

# Compare strategies (using functions from 02)
print("Retrieval Accuracy by Chunking Strategy:\n")
for name, chunks in [
    ("Fixed", chunk_fixed(doc, 60, 10)),
    ("Recursive", chunk_recursive(doc, 60)),
    ("Semantic", chunk_semantic(doc)),
]:
    acc = eval_retrieval(chunks, test_cases)
    bar = "█" * int(acc * 20)
    print(f"  {name:12s}: {acc*100:.0f}% {bar}")
print(f"\nRecursive/Semantic win because they keep related info together.")


### Step 4 · Production Document Pipeline — Complete Class

Load any format, chunk with chosen strategy, embed, return ready-to-index chunks.

**`04_pipeline.py`** · _Block 4 of 4_

COMPLETE DOCUMENT PROCESSING PIPELINE


In [ ]:
# COMPLETE DOCUMENT PROCESSING PIPELINE
import google.generativeai as genai, numpy as np, os, re
from pathlib import Path
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class DocumentPipeline:
    """Load → Clean → Chunk → Embed. Ready for vector DB."""

    def __init__(self, strategy="recursive", chunk_size=200, overlap=50):
        self.strategy = strategy
        self.chunk_size = chunk_size
        self.overlap = overlap

    def _clean(self, text):
        text = re.sub(r'\s+', ' ', text)
        text = re.sub(r'Page \d+ of \d+', '', text)  # remove page numbers
        text = re.sub(r'http\S+', '', text)  # remove URLs
        return text.strip()

    def process(self, text, source="doc"):
        text = self._clean(text)
        if self.strategy == "fixed":
            chunks = chunk_fixed(text, self.chunk_size, self.overlap)
        elif self.strategy == "recursive":
            chunks = chunk_recursive(text, self.chunk_size)
        else:
            chunks = chunk_semantic(text)

        results = []
        for i, c in enumerate(chunks):
            emb = genai.embed_content(model="models/text-embedding-004", content=c)["embedding"]
            results.append({"id": f"{source}_{i}", "text": c, "embedding": emb,
                           "source": source, "words": len(c.split())})
        return results

pipe = DocumentPipeline(strategy="recursive", chunk_size=80)
results = pipe.process(doc, source="netsetos_faq")
print(f"Pipeline: {len(results)} chunks, ready for vector DB")
for r in results:
    print(f"  {r['id']:20s} {r['words']:3d}w  {r['text'][:50]}...")


---

## Wrap-up

You've walked through all 4 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
